In [2]:
import importlib
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.fft import fft, fftfreq
from pathlib import Path
from utils import data_loader_utils

# Isolation Forest
from itertools import product
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

# SVM
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import OneClassSVM
from sklearn.covariance import LedoitWolf

importlib.reload(data_loader_utils)

<module 'utils.data_loader_utils' from '/home/scopellie/EECE5644_Project/utils/data_loader_utils.py'>

## Data Prep

### Initial data visualization / validation of format

In [3]:
data_path = Path("./data").absolute()
file_list = data_loader_utils.find_all_h5s_in_dir(data_path)
print(f"Number of files: {len(file_list)}")

Number of files: 1702


In [4]:
# Test - Reading and formatting first file
file_df = data_loader_utils.format_file(file_list[0])
file_df

,X,Y,Z,timestamp,label,machine,process,i,window
0,-15.0,25.0,-1013.0,0.0,True,1,2,13,0
1,-21.0,19.0,-1013.0,0.5,True,1,2,13,0
2,-17.0,29.0,-1013.0,1.0,True,1,2,13,0
3,-13.0,13.0,-1013.0,1.5,True,1,2,13,0
4,-9.0,25.0,-1020.0,2.0,True,1,2,13,0
...,...,...,...,...,...,...,...,...,...
94203,0.0,23.0,-1013.0,47101.5,True,1,2,13,23
94204,-19.0,11.0,-1018.0,47102.0,True,1,2,13,23
94205,-1.0,31.0,-1015.0,47102.5,True,1,2,13,23
94206,-19.0,9.0,-1020.0,47103.0,True,1,2,13,23


### Extraction of data window features

In [5]:
with h5py.File(file_list[0], 'r') as f:
    dataset = f['vibration_data'][:]

dataset    

array([[-1.500e+01,  2.500e+01, -1.013e+03],
       [-2.100e+01,  1.900e+01, -1.013e+03],
       [-1.700e+01,  2.900e+01, -1.013e+03],
       ...,
       [-1.000e+00,  3.100e+01, -1.015e+03],
       [-1.900e+01,  9.000e+00, -1.020e+03],
       [-2.100e+01,  2.300e+01, -1.016e+03]], shape=(94208, 3))

In [6]:
window_size = 4000
windows = []
df = pd.DataFrame()

for file in file_list[:]:
    with h5py.File(file, 'r') as f:
        dataset = f['vibration_data'][:]
        num_windows = len(dataset) // window_size
        for i in range(0, num_windows):
            w = dataset[i * window_size: (i + 1) * window_size]
            features = data_loader_utils.extract_window_features(w, file.split("/")[-1])
            df_feat = pd.DataFrame([features])
            df_feat["label"] = 1 if file.split("/")[-2] == "good" else -1
            df = pd.concat([df, df_feat], ignore_index=True)

print(df.shape)
df.head(5)

(43847, 102)


,x_mean,x_std,x_min,x_max,x_ptp,x_rms,x_abs_mean,x_skew,x_kurtosis,x_crest_factor,...,mag_spectral_spread,mag_spectral_skew,mag_spectral_kurtosis,mag_spectral_entropy,mag_spectral_rolloff_85,mag_band_power_low,mag_band_power_mid,mag_band_power_high,filename,label
0,-10.63150,289.719079,-1901.0,1477.0,3378.0,289.877887,96.18050,0.252032,13.670715,6.557934,...,197.021001,3.673490,15.467634,4.324752,48.828125,17047.712269,308.506648,1356.958703,M01_Aug_2019_OP02_013.h5,1
1,0.12400,361.933722,-1684.0,1657.0,3341.0,361.888499,155.94600,-0.042713,9.689604,4.653367,...,210.418665,2.741871,9.666535,4.400707,87.890625,49178.504902,1021.077338,7795.512593,M01_Aug_2019_OP02_013.h5,1
2,-4.66625,427.397194,-1963.0,1948.0,3911.0,427.369241,220.13875,-0.205984,5.502912,4.593218,...,232.113597,0.970753,3.092640,4.878282,400.390625,42027.463057,1143.923769,27943.399209,M01_Aug_2019_OP02_013.h5,1
3,0.28850,349.357132,-1772.0,1704.0,3476.0,349.313578,175.08650,-0.193171,7.574259,5.072806,...,221.757508,0.609678,2.634250,5.367646,396.484375,33316.014787,1097.377716,32389.643297,M01_Aug_2019_OP02_013.h5,1
4,-2.20975,333.288878,-2051.0,1809.0,3860.0,333.254540,160.86475,-0.239712,10.389008,6.154455,...,195.345246,2.161938,7.205813,4.322214,396.484375,47439.534665,1304.938348,11556.580952,M01_Aug_2019_OP02_013.h5,1


In [7]:
good_file_windows = df[df["label"] == 1].drop(columns=["filename"])
bad_file_windows = df[df["label"] == -1].drop(columns=["filename"])

good_windows = good_file_windows.shape[0]
bad_windows = bad_file_windows.shape[0]
print(f"Windows with good label: {good_windows}; Windows with bad label: {bad_windows}.")

Windows with good label: 42472; Windows with bad label: 1375.


## Training both models

### Splitting training and testing buckets

Training - Validation - Testing: 80% - 10% - 10%

In [8]:
ratio = 0.8 # General ratio for training bucket

train_ratio = ratio * (1 + (bad_windows / good_windows))
test_ratio = 1 - train_ratio
n_train = round(good_windows * train_ratio,0)
n_test_good = round(good_windows * test_ratio, 0)

print(f"""Training ratio: {n_train / (good_windows + bad_windows)}; 
{n_train} training vectors \n""")
print(f"""Testing ratio: {(n_test_good + bad_windows) / (good_windows + bad_windows)};
{n_test_good + bad_windows} testing vectors, {n_test_good} with "good" label""")

Training ratio: 0.8000091226309668; 
35078.0 training vectors 

Testing ratio: 0.19999087736903323;
8769.0 testing vectors, 7394.0 with "good" label


In [9]:
X_train = good_file_windows.sample(frac=train_ratio, random_state=42)
y_train = X_train[["label"]]
X_train = X_train.drop(columns=["label"])

bad_sorted = df[df["label"] == -1].sort_values(by="filename").drop(columns=["filename"])
bad_windows_sel = bad_sorted[:bad_windows//2]
bad_windows_fin = bad_sorted[bad_windows//2:]

X_test_good = df.drop(X_train.index).drop(bad_file_windows.index).drop(columns=["filename"])
X_test_sel = pd.concat([X_test_good.iloc[:int(n_test_good//2)], bad_windows_sel], ignore_index=True)
X_test_fin = pd.concat([X_test_good.iloc[int(n_test_good//2):], bad_windows_fin], ignore_index=True)

y_test_sel = X_test_sel[["label"]]
X_test_sel = X_test_sel.drop(columns=["label"])
y_test_fin = X_test_fin[["label"]]
X_test_fin = X_test_fin.drop(columns=["label"])

y_anom_sel = (y_test_sel == -1).astype(int)
y_anom_fin = (y_test_fin == -1).astype(int)

In [10]:
maha = make_pipeline(StandardScaler()).fit(X_train)
cov = LedoitWolf().fit(maha.transform(X_train))
test_scores_maha = cov.mahalanobis(maha.transform(X_test_sel))

print(roc_auc_score(y_anom_sel, test_scores_maha),
      average_precision_score(y_anom_sel, test_scores_maha))

0.7604442643805375 0.5281243972750372


### Evaluating features

In [11]:
aucs = {c: roc_auc_score(y_anom_sel, X_test_sel[c]) for c in X_test_sel.columns}
feat_auc = pd.Series(aucs)

# Distance from 0.5 = informative, regardless of direction
strength = feat_auc.sub(0.5).abs().sort_values(ascending=False)

summary = pd.DataFrame({
    "auc": feat_auc[strength.index],
    "strength": strength,
    "direction": np.where(feat_auc[strength.index] > 0.5,
                          "higher in anomalies", "lower in anomalies"),
})
print(summary.head(15))

                            auc  strength            direction
x_band_power_mid       0.934727  0.434727  higher in anomalies
mag_band_power_mid     0.926378  0.426378  higher in anomalies
z_band_power_mid       0.925731  0.425731  higher in anomalies
y_band_power_mid       0.903593  0.403593  higher in anomalies
z_rms                  0.891583  0.391583  higher in anomalies
z_spectral_spread      0.126906  0.373094   lower in anomalies
y_spectral_skew        0.872405  0.372405  higher in anomalies
y_peak_psd             0.871595  0.371595  higher in anomalies
y_spectral_kurtosis    0.861326  0.361326  higher in anomalies
y_spectral_entropy     0.147278  0.352722   lower in anomalies
y_abs_mean             0.851169  0.351169  higher in anomalies
y_zero_crossing_rate   0.160200  0.339800   lower in anomalies
y_kurtosis             0.172647  0.327353   lower in anomalies
y_spectral_rolloff_85  0.173105  0.326895   lower in anomalies
z_spectral_kurtosis    0.824881  0.324881  higher in an

In [12]:
summary["axis"] = [c.split("_")[0] for c in summary.index]
summary["feature"] = ["_".join(c.split("_")[1:]) for c in summary.index]

print(summary.groupby("axis")["strength"].mean().sort_values(ascending=False))
print(summary.groupby("feature")["strength"].mean().sort_values(ascending=False))

keep = strength[strength > 0.1].index

axis
y      0.251963
z      0.230087
mag    0.201291
x      0.073627
Name: strength, dtype: float64
feature
band_power_mid         0.422607
rms                    0.251216
peak_psd               0.247215
zero_crossing_rate     0.233648
shape_factor           0.233261
kurtosis               0.227958
abs_mean               0.220863
spectral_spread        0.220379
total_power            0.220335
std                    0.219830
spectral_entropy       0.214931
spectral_skew          0.187316
mean                   0.177396
spectral_kurtosis      0.172476
max                    0.171457
impulse_factor         0.164393
spectral_rolloff_85    0.164042
ptp                    0.163278
crest_factor           0.154073
spectral_centroid      0.148164
dominant_freq          0.131293
band_power_high        0.122382
min                    0.118800
skew                   0.084014
band_power_low         0.059724
Name: strength, dtype: float64


### One Class SVM parameters sweep

In [13]:
base_gamma = 1 / (X_train[keep].shape[1] * StandardScaler().fit_transform(X_train[keep]).var())

results = []
for g_mult in (5, 10, 20):
    for nu in (0.01, 0.05, 0.1):
        m = make_pipeline(StandardScaler(), OneClassSVM(gamma=base_gamma * g_mult, nu=nu)).fit(X_train[keep])
        s = -m.decision_function(X_test_sel[keep])
        results.append((g_mult, nu,
                        roc_auc_score(y_anom_sel, s),
                        average_precision_score(y_anom_sel, s)))

print(pd.DataFrame(results, columns=["gamma_mult", "nu", "roc_auc", "pr_auc"]).sort_values("pr_auc", ascending=False))

    gamma_mult    nu   roc_auc    pr_auc
16        20.0  0.01  0.937790  0.836227
18        20.0  0.10  0.937727  0.836216
17        20.0  0.05  0.937713  0.836201
19        20.0  0.30  0.937570  0.835476
14        10.0  0.10  0.900522  0.818205
13        10.0  0.05  0.900511  0.818203
12        10.0  0.01  0.900484  0.818161
8          5.0  0.01  0.881649  0.796846
9          5.0  0.05  0.881417  0.794692
15        10.0  0.30  0.893558  0.792690
10         5.0  0.10  0.878375  0.782082
11         5.0  0.30  0.859367  0.726478
4          1.0  0.01  0.816740  0.673397
5          1.0  0.05  0.822366  0.656582
0          0.5  0.01  0.795537  0.637482
6          1.0  0.10  0.817000  0.632513
7          1.0  0.30  0.805761  0.601304
1          0.5  0.05  0.777682  0.587871
2          0.5  0.10  0.785245  0.580101
3          0.5  0.30  0.792630  0.573871


In [20]:
SVM_param = pd.DataFrame(results, columns=["gamma_mult", "nu", "roc_auc", "pr_auc"]).sort_values("pr_auc", ascending=False)
best_g_mult = SVM_param["gamma_mult"].iloc[0]
best_nu = SVM_param["nu"].iloc[0]

### Isolation Forest parameters sweep

In [22]:
results = []
for n_estimators, max_samples, max_features in product((100, 200, 300, 500), (128, 256, 512, 1024, 2048), (0.1, 0.3, 0.5, 0.7, 1.0)):
    scores = []
    m = IsolationForest(
        n_estimators=n_estimators,
        max_samples=min(max_samples, len(X_train[keep])),
        max_features=max_features,
        random_state=42,
        n_jobs=-1,
    ).fit(X_train[keep])
    scores.append(-m.score_samples(X_test_sel[keep]))
    s = np.mean(scores, axis=0)
    results.append((n_estimators, max_samples, max_features, roc_auc_score(y_anom_sel, s), average_precision_score(y_anom_sel, s)))

df_IF = pd.DataFrame(results, columns=["n_estimators", "max_samples", "max_features", "roc_auc", "pr_auc"])
print(df_IF.sort_values("pr_auc", ascending=False))

    n_estimators  max_samples  max_features   roc_auc    pr_auc
8            100          256           0.7  0.813782  0.556303
33           200          256           0.7  0.812280  0.547415
11           100          512           0.3  0.811803  0.538689
38           200          512           0.7  0.809197  0.533708
43           200         1024           0.7  0.797571  0.531288
..           ...          ...           ...       ...       ...
75           500          128           0.1  0.758380  0.442478
9            100          256           1.0  0.781447  0.442298
14           100          512           1.0  0.756614  0.435266
19           100         1024           1.0  0.762162  0.428595
4            100          128           1.0  0.756192  0.416280

[100 rows x 5 columns]


In [23]:
IF_param = df_IF.sort_values("pr_auc", ascending=False)
best_n_estimators = IF_param["n_estimators"].iloc[0]
best_max_samples = IF_param["max_samples"].iloc[0]
best_max_features = IF_param["max_features"].iloc[0]

## Results

In [33]:
ocsvm = make_pipeline(
    StandardScaler(),
    OneClassSVM(kernel="rbf", gamma=base_gamma * best_g_mult, nu=best_nu),
)
ocsvm.fit(X_train[keep])

test_scores_svm = -ocsvm.decision_function(X_test_fin[keep])

print("One Class SVM Results:")
print(f"anomaly rate: {y_anom_fin["label"].mean():<1.4f}")
print(f"ROC-AUC:      {roc_auc_score(y_anom_fin, test_scores_svm):<1.4f}")
print(f"PR-AUC :      {average_precision_score(y_anom_fin, test_scores_svm):<1.4f}")

One Class SVM Results:
anomaly rate: 0.1569
ROC-AUC:      0.9725
PR-AUC :      0.9087


In [32]:
clf = IsolationForest(contamination='auto', n_estimators=best_n_estimators, max_samples=best_max_samples, max_features=best_max_features, random_state=42)
clf.fit(X_train[keep])

test_scores_if = -clf.score_samples(X_test_fin[keep])

print("Isolation Forest Results:")
print(f"anomaly rate: {y_anom_fin["label"].mean():<1.4f}")
print(f"ROC-AUC:      {roc_auc_score(y_anom_fin, test_scores_if):<1.4f}")
print(f"PR-AUC :      {average_precision_score(y_anom_fin, test_scores_if):<1.4f}")

Isolation Forest Results:
anomaly rate: 0.1569
ROC-AUC:      0.8299
PR-AUC :      0.6142
